# WCT: Universal Style Transfer via Feature Transforms

**Paper**: [Universal Style Transfer via Feature Transforms (Li et al., 2017)](https://arxiv.org/abs/1705.08086)

WCT performs style transfer by matching the full covariance structure of content features to style features, capturing richer style patterns than mean/std alignment alone.

**Architecture**: VGG Encoder (frozen) → WCT Transform → Decoder (trainable)

In [1]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch

/Users/preethiraghuraman/Documents/School/Winter 2025/Deployment of Deep Learning Models/Lectures/Tutorial/venv_mlf/lib/python3.14/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


## Setup & Imports

In [2]:
# !pip install torch torchvision pillow matplotlib numpy tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os
import glob
from tqdm import tqdm

# Set device - prefer MPS (Apple Silicon) > CUDA > CPU
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f"Using device: {device}")

# For reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Paths to your image directories
CONTENT_DIR = '../images/val2017'
STYLE_DIR = '../images/style'

Using device: mps


In [3]:
import os
print(os.getcwd())

print("Content exists:", os.path.exists(CONTENT_DIR))
print("Style exists:", os.path.exists(STYLE_DIR))

/Users/preethiraghuraman/cs679-finalproject/phase3
Content exists: True
Style exists: True


## Part 1: Utility Functions

Channel-wise mean and standard deviation, used by the WCT loss.

In [4]:
def calc_mean_std(feat, eps=1e-5):
    """
    Calculate channel-wise mean and std.
    
    Args:
        feat: (B, C, H, W) feature maps
        eps: Small value for numerical stability
    
    Returns:
        mean: (B, C, 1, 1)
        std: (B, C, 1, 1)
    """
    batch_size, num_channels = feat.size()[:2]
    
    # Calculate statistics across spatial dimensions (H, W)
    feat_var = feat.view(batch_size, num_channels, -1).var(dim=2) + eps
    feat_std = feat_var.sqrt().view(batch_size, num_channels, 1, 1)
    feat_mean = feat.view(batch_size, num_channels, -1).mean(dim=2).view(batch_size, num_channels, 1, 1)
    
    return feat_mean, feat_std


def adain(content_feat, style_feat):
    """
    Adaptive Instance Normalization.
    
    Args:
        content_feat: Content features (B, C, H, W)
        style_feat: Style features (B, C, H, W)
    
    Returns:
        AdaIN output with content's structure and style's statistics
    """
    assert content_feat.size()[:2] == style_feat.size()[:2]
    
    # Get statistics
    content_mean, content_std = calc_mean_std(content_feat)
    style_mean, style_std = calc_mean_std(style_feat)
    
    # Normalize content (remove its style)
    normalized = (content_feat - content_mean) / content_std
    
    # Apply style statistics
    return normalized * style_std + style_mean

## Part 2: VGG Encoder

The encoder extracts features from images using pretrained VGG-19 (frozen).

**Architecture**: VGG-19 up to relu4_1
- Input: (B, 3, H, W)
- Output: (B, 512, H/8, W/8)
- Pretrained on ImageNet, **frozen** (not trained)

In [5]:
class VGGEncoder(nn.Module):
    """
    VGG-19 encoder (frozen, pretrained on ImageNet).
    Extracts features up to relu4_1.
    """
    
    def __init__(self):
        super().__init__()
        
        # Load pretrained VGG-19
        vgg = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1).features
        
        # VGG-19 feature layers:
        #   0-3:   conv1 block (relu1_1 at index 1)
        #   4:     maxpool (/2)
        #   5-10:  conv2 block (relu2_1 at index 6)
        #   10:    maxpool (/4) -- included at end of slice2
        #   11-17: conv3 block (relu3_1 at index 11)
        #   18:    maxpool (/8) -- included at start of slice4
        #   19:    conv2d(256, 512), 20: relu  ← relu4_1
        self.slice1 = nn.Sequential(*list(vgg[:4]))    # relu1_1
        self.slice2 = nn.Sequential(*list(vgg[4:11]))  # relu2_1
        self.slice3 = nn.Sequential(*list(vgg[11:18])) # relu3_1
        self.slice4 = nn.Sequential(*list(vgg[18:21])) # relu4_1 (pool + conv + relu)
        
        # Freeze all parameters
        for param in self.parameters():
            param.requires_grad = False
    
    def forward(self, x, output_last_only=True):
        """
        Extract features.
        
        Args:
            x: Input image (B, 3, H, W)
            output_last_only: If True, return only relu4_1
                             If False, return dict with all layers
        
        Returns:
            If output_last_only: (B, 512, H/8, W/8)
            Else: dict with relu1_1, relu2_1, relu3_1, relu4_1
        """
        h1 = self.slice1(x)
        h2 = self.slice2(h1)
        h3 = self.slice3(h2)
        h4 = self.slice4(h3)
        
        if output_last_only:
            return h4
        else:
            return {
                'relu1_1': h1,
                'relu2_1': h2,
                'relu3_1': h3,
                'relu4_1': h4
            }

# Test encoder
encoder = VGGEncoder().to(device)
encoder.eval()

test_img = torch.randn(1, 3, 256, 256).to(device)
features = encoder(test_img)
print(f"Encoder test:")
print(f"  Input: {test_img.shape}")
print(f"  Output: {features.shape}")  # Should be (1, 512, 32, 32)
print(f"  Encoder working!")

Encoder test:
  Input: torch.Size([1, 3, 256, 256])
  Output: torch.Size([1, 512, 32, 32])
  Encoder working!


## Part 3: Decoder

The decoder inverts features back to image space.

**Key points**:
- Mirrors encoder structure (but inverted)
- Uses upsampling instead of pooling
- **NO normalization layers** (important!)
- Uses reflection padding to reduce artifacts

In [6]:
class Decoder(nn.Module):
    """
    Decoder that inverts VGG features to image space.
    NO normalization layers!
    """
    
    def __init__(self):
        super().__init__()
        
        # Decoder mirrors encoder but reversed
        # 512 -> 256 -> 128 -> 64 -> 3 channels
        
        self.decoder = nn.Sequential(
            # Block 4: 512 -> 256, upsample 2x
            nn.ReflectionPad2d(1),
            nn.Conv2d(512, 256, 3),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='nearest'),
            
            # Block 3: 256 -> 256 -> 256 -> 128, upsample 2x
            nn.ReflectionPad2d(1),
            nn.Conv2d(256, 256, 3),
            nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(256, 256, 3),
            nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(256, 256, 3),
            nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(256, 128, 3),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='nearest'),
            
            # Block 2: 128 -> 64, upsample 2x
            nn.ReflectionPad2d(1),
            nn.Conv2d(128, 128, 3),
            nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(128, 64, 3),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='nearest'),
            
            # Block 1: 64 -> 3
            nn.ReflectionPad2d(1),
            nn.Conv2d(64, 64, 3),
            nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(64, 3, 3)
        )
    
    def forward(self, x):
        return self.decoder(x)

# Test decoder
decoder = Decoder().to(device)
test_feat = torch.randn(1, 512, 32, 32).to(device)
output = decoder(test_feat)
print(f"Decoder test:")
print(f"  Input: {test_feat.shape}")
print(f"  Output: {output.shape}")
print(f"  ✓ Decoder working!")

Decoder test:
  Input: torch.Size([1, 512, 32, 32])
  Output: torch.Size([1, 3, 256, 256])
  ✓ Decoder working!


## Part 4: Image Utilities

In [7]:
# ImageNet normalization
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

def load_image(path, size=None):
    """
    Load and preprocess image.
    
    Args:
        path: Image path
        size: Resize to (size, size) if specified
    
    Returns:
        Preprocessed tensor (3, H, W)
    """
    img = Image.open(path).convert('RGB')
    
    if size:
        img = img.resize((size, size), Image.LANCZOS)
    
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD)
    ])
    
    return transform(img)


def denormalize(tensor):
    """
    Denormalize from ImageNet stats.
    """
    mean = torch.tensor(MEAN).view(-1, 1, 1)
    std = torch.tensor(STD).view(-1, 1, 1)
    
    if tensor.dim() == 4:
        mean = mean.unsqueeze(0).to(tensor.device)
        std = std.unsqueeze(0).to(tensor.device)
    else:
        mean = mean.to(tensor.device)
        std = std.to(tensor.device)
    
    return tensor * std + mean


def tensor_to_image(tensor):
    """
    Convert tensor to PIL Image.
    """
    if tensor.dim() == 4:
        tensor = tensor[0]
    
    tensor = denormalize(tensor).clamp(0, 1)
    img = transforms.ToPILImage()(tensor.cpu())
    return img


def show_images(images, titles=None, figsize=(15, 5)):
    """
    Display multiple images in a row.
    
    Args:
        images: List of tensors or PIL Images
        titles: Optional list of titles
    """
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    
    if n == 1:
        axes = [axes]
    
    for i, (ax, img) in enumerate(zip(axes, images)):
        if isinstance(img, torch.Tensor):
            img = tensor_to_image(img)
        
        ax.imshow(img)
        ax.axis('off')
        
        if titles and i < len(titles):
            ax.set_title(titles[i])
    
    plt.tight_layout()
    plt.show()

print("Image utilities defined")

Image utilities defined


## Part 5: Dataset & DataLoader

### Dataset

You need:
1. **Content images**: MS-COCO dataset (~80K images) or your own photos
2. **Style images**: WikiArt paintings (~80K images) or your own artworks

During training, we randomly pair content and style images.

In [8]:
class StyleTransferDataset(Dataset):
    """
    Dataset that pairs content with random style images.
    """
    
    def __init__(self, content_dir, style_dir, crop_size=256):
        self.content_paths = self._get_paths(content_dir)
        self.style_paths = self._get_paths(style_dir)
        
        print(f"Found {len(self.content_paths)} content images and {len(self.style_paths)} style images")
        
        self.transform = transforms.Compose([
            transforms.Resize(512),
            transforms.RandomCrop(crop_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=MEAN, std=STD)
        ])
    
    def _get_paths(self, directory):
        paths = []
        for ext in ['*.jpg', '*.jpeg', '*.png', '*.webp', '*.avif']:
            paths.extend(glob.glob(os.path.join(directory, '**', ext), recursive=True))
        return sorted(paths)
    
    def __len__(self):
        return len(self.content_paths)
    
    def __getitem__(self, idx):
        # Load content
        content = Image.open(self.content_paths[idx]).convert('RGB')
        
        # Randomly select style
        style_idx = np.random.randint(len(self.style_paths))
        style = Image.open(self.style_paths[style_idx]).convert('RGB')
        
        return self.transform(content), self.transform(style)

dataset = StyleTransferDataset(CONTENT_DIR, STYLE_DIR, crop_size=256)

train_loader = DataLoader(dataset, batch_size=4, shuffle=True)

Found 5059 content images and 9 style images


## Part 6: WCT Transform

The Whitening and Coloring Transform matches the full covariance structure of features.

In [9]:
def compute_covariance(feat, eps=1e-5):
    """
    Compute covariance matrix for flattened features.

    Args:
        feat: (C, N) tensor
        eps: Numerical stability constant

    Returns:
        (C, C) covariance matrix
    """
    C, N = feat.shape
    cov = feat @ feat.t() / (N - 1)
    cov = cov + eps * torch.eye(C, device=feat.device, dtype=feat.dtype)
    return cov


def whiten_and_color_single(content_feat, style_feat, eps=1e-5):
    """
    Apply Whitening and Coloring Transform to one content-style pair.

    Args:
        content_feat: (C, H, W)
        style_feat: (C, H, W)
        eps: Numerical stability constant

    Returns:
        Transformed features (C, H, W)
    """
    C, H, W = content_feat.shape
    original_device = content_feat.device

    # Move to CPU for numerically stable eigendecomposition
    content_feat = content_feat.float().cpu()
    style_feat = style_feat.float().cpu()

    # Flatten spatial dimensions
    content_flat = content_feat.view(C, -1)
    style_flat = style_feat.view(C, -1)

    # Compute means
    content_mean = content_flat.mean(dim=1, keepdim=True)
    style_mean = style_flat.mean(dim=1, keepdim=True)

    # Center features
    content_centered = content_flat - content_mean
    style_centered = style_flat - style_mean

    # Covariances
    content_cov = compute_covariance(content_centered, eps=eps)
    style_cov = compute_covariance(style_centered, eps=eps)

    # Eigendecomposition (on CPU for numerical stability)
    c_evals, c_evecs = torch.linalg.eigh(content_cov)
    s_evals, s_evecs = torch.linalg.eigh(style_cov)

    # Keep stable eigenvalues only (use larger threshold for stability)
    k_c = (c_evals > 1e-3).sum()
    k_s = (s_evals > 1e-3).sum()

    c_evals = c_evals[-k_c:]
    c_evecs = c_evecs[:, -k_c:]

    s_evals = s_evals[-k_s:]
    s_evecs = s_evecs[:, -k_s:]

    # Whitening transform
    whitening = c_evecs @ torch.diag(c_evals.rsqrt()) @ c_evecs.t()
    whitened = whitening @ content_centered

    # Coloring transform
    coloring = s_evecs @ torch.diag(s_evals.sqrt()) @ s_evecs.t()
    transformed = coloring @ whitened + style_mean

    return transformed.view(C, H, W).to(original_device)


def wct(content_feat, style_feat, alpha=1.0, eps=1e-5):
    """
    Batch Whitening and Coloring Transform.

    Args:
        content_feat: (B, C, H, W)
        style_feat: (B, C, H, W)
        alpha: Style strength
        eps: Numerical stability constant

    Returns:
        WCT-transformed features (B, C, H, W)
    """
    assert content_feat.size()[:2] == style_feat.size()[:2]

    B, C, H, W = content_feat.shape
    outputs = []

    for b in range(B):
        transformed = whiten_and_color_single(content_feat[b], style_feat[b], eps=eps)
        transformed = alpha * transformed + (1 - alpha) * content_feat[b]
        outputs.append(transformed)

    return torch.stack(outputs, dim=0)

print("\u2713 WCT utilities defined")

✓ WCT utilities defined


## Part 7: WCT Network

In [10]:
class WCTNet(nn.Module):
    """
    Complete WCT network for arbitrary style transfer.
    Uses a frozen VGG encoder and a trainable decoder.
    """

    def __init__(self):
        super().__init__()

        self.encoder = VGGEncoder()
        self.decoder = Decoder()

        # Encoder is frozen, only decoder trains
        self.encoder.eval()

    def encode(self, x):
        return self.encoder(x)

    def decode(self, x):
        return self.decoder(x)

    def forward(self, content, style, alpha=1.0, eps=1e-5):
        """
        Style transfer using WCT.

        Args:
            content: Content images (B, 3, H, W)
            style: Style images (B, 3, H, W)
            alpha: Style strength
            eps: Numerical stability constant

        Returns:
            Stylized images (B, 3, H, W)
        """
        content_feat = self.encode(content)
        style_feat = self.encode(style)

        t = wct(content_feat, style_feat, alpha=alpha, eps=eps)

        return self.decode(t)

    def forward_with_features(self, content, style, alpha=1.0, eps=1e-5):
        """
        Same as forward but also returns transformed features.
        """
        content_feat = self.encode(content)
        style_feat = self.encode(style)

        t = wct(content_feat, style_feat, alpha=alpha, eps=eps)
        output = self.decode(t)

        return output, t

# Create model
wct_model = WCTNet().to(device)
print(f"WCT model created!")
print(f"  Encoder params: {sum(p.numel() for p in wct_model.encoder.parameters()):,} (frozen)")
print(f"  Decoder params: {sum(p.numel() for p in wct_model.decoder.parameters()):,} (trainable)")

WCT model created!
  Encoder params: 3,505,728 (frozen)
  Decoder params: 3,505,219 (trainable)


## Part 8: Covariance Helpers

In [11]:
def compute_batched_covariance(feat, eps=1e-5):
    """
    Compute covariance matrices for batched features.

    Args:
        feat: (B, C, H, W)

    Returns:
        covariances: (B, C, C)
    """
    B, C, H, W = feat.shape
    covs = []

    for b in range(B):
        x = feat[b].view(C, -1)
        x = x - x.mean(dim=1, keepdim=True)
        cov = x @ x.t() / (x.shape[1] - 1)
        cov = cov + eps * torch.eye(C, device=x.device, dtype=x.dtype)
        covs.append(cov)

    return torch.stack(covs, dim=0)

print("✓ Covariance helpers defined")

✓ Covariance helpers defined


## Part 9: WCT Loss

In [12]:
class WCTLoss(nn.Module):
    """
    Loss = content_loss + λ * style_loss

    Style loss matches:
    - mean
    - std
    - covariance
    """

    def __init__(self, style_weight=10.0, cov_weight=1.0):
        super().__init__()
        self.style_weight = style_weight
        self.cov_weight = cov_weight
        self.encoder = VGGEncoder()
        self.encoder.eval()
        self.mse = nn.MSELoss()

    def content_loss(self, output_feat, target_feat):
        """MSE between output and WCT target features."""
        return self.mse(output_feat, target_feat)

    def mean_std_loss_layer(self, output_feat, style_feat):
        """Match mean and std for one layer."""
        out_mean, out_std = calc_mean_std(output_feat)
        style_mean, style_std = calc_mean_std(style_feat)

        return self.mse(out_mean, style_mean) + self.mse(out_std, style_std)

    def covariance_loss_layer(self, output_feat, style_feat):
        """Match covariance for one layer."""
        out_cov = compute_batched_covariance(output_feat)
        style_cov = compute_batched_covariance(style_feat)

        return self.mse(out_cov, style_cov)

    def style_loss(self, output, style):
        """Total style loss across multiple layers."""
        output_feats = self.encoder(output, output_last_only=False)
        style_feats = self.encoder(style, output_last_only=False)

        loss = 0
        for layer in ['relu1_1', 'relu2_1', 'relu3_1', 'relu4_1']:
            loss_mean_std = self.mean_std_loss_layer(output_feats[layer], style_feats[layer])
            loss_cov = self.covariance_loss_layer(output_feats[layer], style_feats[layer])
            loss += loss_mean_std + self.cov_weight * loss_cov

        return loss

    def forward(self, output, style, target_feat):
        """
        Compute total loss.

        Args:
            output: Generated image
            style: Style image
            target_feat: WCT output features (t)

        Returns:
            (total_loss, content_loss, style_loss)
        """
        output_feat = self.encoder(output)
        loss_c = self.content_loss(output_feat, target_feat)

        loss_s = self.style_loss(output, style)

        loss_total = loss_c + self.style_weight * loss_s

        return loss_total, loss_c, loss_s

# Test loss
wct_criterion = WCTLoss(style_weight=10.0, cov_weight=1.0).to(device)
print("✓ WCT loss function defined")

✓ WCT loss function defined


## Part 10: WCT Training

In [ ]:
def train_wct(model, train_loader, num_epochs=60, lr=1e-4, style_weight=10.0, cov_weight=1.0):
    """
    Train WCT network.

    Args:
        model: WCTNet model
        train_loader: DataLoader with content-style pairs
        num_epochs: Number of epochs
        lr: Learning rate
        style_weight: Weight for style loss
        cov_weight: Weight for covariance term inside style loss
    """
    optimizer = optim.Adam(model.decoder.parameters(), lr=lr)
    criterion = WCTLoss(style_weight=style_weight, cov_weight=cov_weight).to(device)

    model.train()

    for epoch in range(num_epochs):
        epoch_loss = 0
        pbar = tqdm(train_loader, desc=f'WCT Epoch {epoch+1}/{num_epochs}')

        for content, style in pbar:
            content = content.to(device)
            style = style.to(device)

            # Forward
            output, t = model.forward_with_features(content, style)

            # Loss
            loss_total, loss_c, loss_s = criterion(output, style, t)

            # Backward
            optimizer.zero_grad()
            loss_total.backward()
            optimizer.step()

            epoch_loss += loss_total.item()
            pbar.set_postfix({
                'loss': f'{loss_total.item():.4f}',
                'content': f'{loss_c.item():.4f}',
                'style': f'{loss_s.item():.4f}'
            })

        avg_loss = epoch_loss / len(train_loader)
        print(f'Epoch {epoch+1}: Avg Loss = {avg_loss:.4f}')

        # Save checkpoint every 20 epochs
        if (epoch + 1) % 20 == 0:
            os.makedirs('checkpoints', exist_ok=True)
            torch.save(model.decoder.state_dict(), f'checkpoints/wct_decoder_epoch{epoch+1}.pth')

    return model

print("\nStarting WCT training...")
wct_model = train_wct(
    wct_model,
    train_loader,
    num_epochs=60,
    lr=1e-4,
    style_weight=10.0,
    cov_weight=1.0
)

os.makedirs('checkpoints', exist_ok=True)
torch.save(wct_model.decoder.state_dict(), 'checkpoints/wct_decoder.pth')
print("Model saved to checkpoints/wct_decoder.pth")


Starting WCT training...


WCT Epoch 1/60:  12%|█▏        | 3/25 [00:02<00:14,  1.49it/s, loss=700.4535, content=20.4631, style=67.9990]

## Part 11: WCT Inference

In [ ]:
def stylize_single_wct(model, content_path, style_path, alpha=1.0):
    """
    Single style transfer using WCT.

    Returns:
        (content_img, style_img, output_img)
    """
    model.eval()

    content = load_image(content_path, size=512).unsqueeze(0).to(device)
    style = load_image(style_path, size=512).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(content, style, alpha=alpha)

    return (
        tensor_to_image(content),
        tensor_to_image(style),
        tensor_to_image(output)
    )

print("WCT inference function defined")

In [ ]:
content_img, style_img, output_img = stylize_single_wct(
    wct_model,
    os.path.join(CONTENT_DIR, 'river.jpg'),
    os.path.join(STYLE_DIR, 'starry_night.jpg'),
    alpha=1.0
)

show_images(
    [content_img, style_img, output_img],
    titles=['Content', 'Style', 'WCT Output']
)

In [ ]:
# Compare different alpha values for WCT
# alpha controls style strength: 0.0 = original content, 1.0 = full stylization
alphas = [0.0, 0.25, 0.5, 0.75, 1.0]

content_path = os.path.join(CONTENT_DIR, 'river.jpg')
style_path = os.path.join(STYLE_DIR, 'starry_night.jpg')

outputs = []
for a in alphas:
    _, _, output_img = stylize_single_wct(
        wct_model,
        content_path,
        style_path,
        alpha=a
    )
    outputs.append(output_img)

show_images(
    outputs,
    titles=[f'alpha={a}' for a in alphas],
    figsize=(20, 5)
)